# Kalman Filter Price-Target Model — Fused Panel with a Per-ISIN Latent (v4)

**Notebook form of `pymc_kalman_filter_pt.py`** (0.9.9.14), aligned with
`probabilistic_ml_model/pymc_models/KalmanFilterModel.py`. Supersedes
`pymc_kalman_filter_pt_v3.ipynb`, which tracked 0.9.9.11.

The cross-sectional spine is the **fused hierarchical panel model**
(`build_fused_kalman_pt_model`):

- **Model B spine** — a rank-1 Intrinsic Coregionalization Model (ICM) over the
  `(isin, time, y_series)` response tensor: the `D` response series share the latent
  per-ISIN factor `mu_isin` through per-series loadings `mu_isin_loading` (primary
  anchored at 1) with a per-series noise diagonal `sigma_series`. **The ICM is dormant
  while `D == 1`** (the default) — `KalmanRunConfig.panel_response_extra` is what
  activates it. Per-time levels are **T direct per-time intercepts** (`alpha_level`);
  the per-series time slope `beta_t` / `beta_slope` is materialised **only** when
  `t_scaled` genuinely varies across ISINs, which the `np.tile`-built T>1 axis does not.
- **Model A refinement** — the risk-aware `expected_return` latent is the structural
  mean, with the heteroscedastic scale `sigma_isin = sigma_base · (1 + cv) / √n` and the
  additive systematic-risk / size / volume tilts
  `risk_adj_return = ER − risk_loading·z(avg_beta) − size_loading·z(mcap_ratio) − volume_loading·z(rel_volume)`.
- **Per-ISIN random intercept** (`sigma_isin_level` × a non-centred `ZeroSumNormal`),
  restored for **`T > 1` only**. It was dropped in an earlier release as non-identified —
  correct *for the T=1 cross-section*; `T` repeated observations per name identify the
  *effect* in the ordinary way. Its **scale is a fixed constant, not a learned parameter**
  — see §0b. `state_path` (isin, time) and `state_now = state_path[:, -1]` are always
  emitted, so the decision latent has exactly one name.

## What's new in v4 (CHANGELOG 0.9.9.13 / 0.9.9.14)

1. **`state_now` is the decision latent.** Every consumer — screen, price-target Monte
   Carlo, risk book, analytics export, §13b plots, prior predictive — resolves it through
   `KALMAN_SCREEN_LATENT` / `resolve_screen_latent`, with a documented fallback to
   `risk_adj_return` for pre-0.9.9.14 NetCDF artifacts. `achieve_prob = sigmoid(state_now)`.
2. **The genuine T=4 panel is now the DEFAULT** (`panel_lookbacks=('6m','3m','1m')`).
   v3 had this backwards: it was the opt-in there, and the collapsed T=1 snapshot is now
   the opt-*out*.
3. **Per-ISIN latent fixes a ~√T over-confidence.** Broadcasting a time-constant `mu_isin`
   across the T lookback slices treated each name's `T` serially-correlated observations as
   `T` iid draws. Re-measured on the 2026-08-16 6,487-name T=4 panel against a no-latent
   baseline: `sigma_base` 0.1519 → **0.1484** and mean per-name posterior sd 0.0154 →
   **0.0889** (5.8×). Both moved in the predicted direction, though by less than the
   0.9.9.14 note recorded (0.3373 → 0.2871 and 0.0545 → 0.2694).
4. **AR(1) time-varying state — tried, measured, rejected, retained.**
   `state_innovation_scale` defaults to `0.0`. See §0b for the evidence.
5. **Exported values corrected twice.** The de-standardisation fix and the log-space
   `expm1` clip (`LOG_UPLIFT_CLIP_*`). Note the de-standardisation delta is **signed**, not
   a uniform overstatement: on the 2026-08-16 panel it runs −1.28 pp at latent −0.5 through
   **+2.26 pp** at +1.0.
6. **Drift design pruned 21 → 15 columns** (condition number 1,580 → 23, max VIF 162 → 3.8)
   and `region` dropped from the crossed group effects.
7. **§9b model comparison** (`run_model_comparison`) closes the last `❌` in this module's
   Bayesian-workflow coverage row.
8. **Artifact export** lands in a per-section subdirectory tree (0.9.9.13); this notebook
   wires `enable_artifact_export()` + `set_export_section()` behind an opt-in flag.

## 2026-08-16 — Phase A review fixes (shipped and verified)

A review of the 2026-08-15 run found two defects reaching the GEIB dashboard and one
schema-integrity gap. All are fixed, and the analytics schema has been re-exported:
**every table now carries `run_id = 8538a5d6ee25`.**

1. **The out-of-support detector was one-sided** (§10c). It tested only the `+500 %` cap
   and never the `−95 %` floor, so names whose entire draw set pinned at the floor shipped
   `expected_sharpe_ratio = −4.28e15` unflagged — the cap test matched **0 of 6,487** rows.
   The test is now symmetric (`er_p05` vs cap, `er_p95` vs floor) and reports each
   direction. It now flags **4 names, all at the floor**: Kioxia, Yuanjie Semiconductor,
   AXTI, SNDK.
2. **No finite guard on the ranking ratios** (§10b). `RiskBookModel` guarded its
   denominators with `> 0`, which a denormal `er_sd` of ~4e−16 passes. All three ratios now
   floor at `MIN_RATIO_DENOMINATOR = 1e-4`. `expected_sharpe_ratio` now spans
   **−33.73 … 7.998** instead of reaching −4.28e15; real values are unchanged.
3. **The analytics schema served two vintages — and the cause was structural, not a
   one-off.** `scripts/export_kalman_analytics.py` never called `export_all_artifacts()`,
   so the production path wrote **2 of the 7 curated tables by construction**
   (`kalman_filtered_price_targets` + `09_diagnostics_01_table`), leaving the other five on
   whatever fit last touched them. The divergence therefore returned on *every* refresh,
   not just on 2026-08-15. The script now calls it, and every frame carries `run_id` /
   `exported_at`. Cross-table `er_mean` disagreement: **6,425 of 6,427 → 0 of 6,487.**
4. **Three exported columns were misnamed for what they measure** — see §10b/§10c. No
   column was renamed (the GEIB contract is unchanged); the `COMMENT ON COLUMN` text and
   the dashboard labels now say what the numbers actually are.
5. **The Kelly card sized on the wrong denominator.** `dashboards/geib/charts/kelly.py`
   used `abs(cvar_5pct_kalman)` as the loss leg of the odds ratio; that column is a
   positive return level, so `b` had a median of 1.28 with an sd of 23.7. It now reuses the
   risk book's own `tail_risk` definition (median 4.27, sd 2.77). **This changes allocation
   values, not just labels.**

> **Two bugs were introduced by the fixes themselves and then caught.** Recorded because
> both are easy to reproduce:
> - `check_export_vintage()` probed each table with `SELECT run_id` and caught the error.
>   On PostgreSQL an `UndefinedColumn` **aborts the transaction**, so the first unstamped
>   table poisoned the connection and every later query failed with
>   `InFailedSqlTransaction` — it reported all eight tables unstamped, *including the two
>   that were correctly stamped*. It now resolves stamped tables from `information_schema`
>   first. Never drive control flow with a speculative `SELECT` on Postgres.
> - Its summary line counted only *stamped* tables, so it printed `OK: single vintage`
>   while five frames sat two days stale. A table that exists without `run_id` is a
>   different vintage, not an unknown one.

> **Still open (Phase B).** Three posterior-predictive statistics reject this model — the
> PIT ECDF (p = 0.00), and the `mean` and `std` t-statistics, whose observed values both
> fall outside the replicated mass. `nu` sits pinned at its 2.5 floor (2.5259 on this run).
> See §6/§8.

## §0 — Environment, imports & run configuration

`PYTENSOR_FLAGS` must be set **before** PyTensor/PyMC are first imported (forward slashes
in any `cxx` path — the flag parser is posix shlex and strips backslashes). Importing
`probabilistic_ml_model` normalises the flags via `force_python_vm()`; `set_env.ps1`
handles the full setup. The C backend is **off** project-wide unless
`PML_ENABLE_PYTENSOR_C=1` is set before import.

All workflow knobs live on **`KalmanRunConfig`** (frozen dataclass).
`KalmanRunConfig.from_env()` resolves only five variables — `RANDOM_SEED`,
`KALMAN_PT_RESULTS_DIR`, `KALMAN_PT_EXPORT_DRAWS`, `PML_FIG_WIDTH_PX`, `LOG_LEVEL`.
Everything else (sampling budget, panel geometry, Monte-Carlo screen, CVaR book,
universe-query dates) keeps its dataclass default and is overridden with
`dataclasses.replace` — never by mutation.

In [ ]:
import logging
from dataclasses import replace

import pymc as pm
from sqlalchemy import create_engine

# Section functions live in the module (single source of truth) — import, don't re-define.
import pymc_kalman_filter_pt as kf
from pymc_kalman_filter_pt import (
    # --- config / plumbing ---
    KalmanRunConfig, get_run_config, set_run_config,
    setup_plotting, resolve_db_url,
    enable_artifact_export, set_export_section, get_export_state,
    # --- §1 data + roles ---
    load_kalman_df, load_feature_catalogue, resolve_feature_roles,
    # --- §2/§3/§4 EDA -> features -> panel ---
    run_eda, map_state_space_features, prepare_kalman_panel_inputs,
    KALMAN_PANEL_RESPONSE_EXTRA,
    # --- §5b -> §9b model / inference / diagnostics ---
    build_panel_model, run_prior_predictive, sample_posterior,
    run_posterior_predictive, run_diagnostics, run_model_comparison,
    present_group_effects,
    # --- §10 -> §10c decision layer ---
    KALMAN_SCREEN_LATENT, resolve_screen_latent,
    UPLIFT_CLIP_LO, UPLIFT_CLIP_HI,
    summarize_panel_screen, compute_cvar_aware_book, export_analytics,
    # --- §10K -> §13 side fits ---
    run_universe_kalman_fit,
    run_single_isin_filter, run_single_isin_stochastic_vol,
    run_mingled_cohort_filter, run_mingled_cohort_stochastic_vol,
    run_granular_forest, run_granular_further_views,
    # --- §14 / §14.1 summary + screen visuals ---
    run_summary, run_recommendations,
    plot_screen_overview, plot_risk_return_scatter, plot_top_candidate_forest,
)

cfg = KalmanRunConfig.from_env()
logging.basicConfig(level=cfg.log_level)
setup_plotting()
engine = create_engine(resolve_db_url())
print('Setup complete — orchestrating:', kf.__file__)
cfg

### §0b — Run toggles

**The genuine `(isin, time)` T=4 panel is the DEFAULT.** `panel_lookbacks=('6m','3m','1m')`
builds a real log-uplift history from the `price_target_{lb}_ago` / `price_{lb}_ago` trails,
with the current snapshot as the final step. Collapse to the T=1 cross-section with
`replace(cfg, panel_lookbacks=())` — faster, no time axis, and the per-ISIN intercept is
not created (it is non-identified there).

| knob                             | default            | what it does / why                                                                                                                                                                                                                                                                                                                                                                             |
|----------------------------------|--------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `panel_lookbacks`                | `('6m','3m','1m')` | T=4 genuine panel. `()` → T=1 snapshot.                                                                                                                                                                                                                                                                                                                                                        |
| `draws`                          | `2000`             | Sampling budget. **Measured on the 2026-08-15 full run: 0 divergences, max R-hat 1.0041, min bulk ESS 824, min tail ESS 1,413 — every parameter clears the 400 gate with room.** (An earlier note here said this budget "landed near 300"; that is stale.) ≈33 min per T=4 run.                                                                                                                |
| `state_innovation_scale`         | `0.0` (**off**)    | AR(1) time-varying state on top of the intercept. Tried and **rejected** at T=4: +0.013 recovery correlation (0.964 vs 0.951) for min ESS **14 vs 69** and max R-hat 1.13 vs 1.03, with `sigma_state`/`rho` drifting between draw budgets (0.265→0.216, 0.63→0.49) — the "two fits disagree" signature of a non-identified variance component. Set `0.1` to enable; revisit on a longer panel. |
| `isin_level_scale` (builder arg) | `0.10`             | **The FIXED scale of the per-ISIN random intercept — a constant, not a prior on a sampled parameter.** `sigma_isin_level` is emitted as `pm.Deterministic(pt.constant(...))`, so it reports sd 0, ESS == total draws and R-hat NaN by construction. See the note below. `0.0` pins the whole layer off (the pre-0.9.9.14 baseline).                                                            |
| `enable_model_comparison`        | `False`            | §9b. Refits both arms **and** computes a pointwise `log_likelihood` per arm (~820 MB each at full panel size) → ≈3× sampling cost.                                                                                                                                                                                                                                                             |
| `comparison_max_isins`           | `800`              | ISIN subsample for §9b; the retained fraction is printed so a truncated comparison never reads as a full one.                                                                                                                                                                                                                                                                                  |
| `panel_response_extra`           | `()`               | Keys of `KALMAN_PANEL_RESPONSE_EXTRA` promoting a **second** response series (`D > 1`) — which is what activates the otherwise-dormant rank-1 ICM. The one supplied series, `pt_dispersion`, drops the collinear drift predictor `feat_pt_noise_drift`. **Off by default:** the `D > 1` path produced the historic R-hat 4.45 / min-ESS 4.3 freeze.                                            |

> **Why `sigma_isin_level` is fixed rather than learned** (`KalmanFilterModel.py:2698-2723`).
> A learned scale was tried first and behaved exactly like the `sigma_group` scalar this
> model already abandoned: on the 2026-08-10 full-scale run (6,540 ISINs, T=4) it was the
> **worst-mixing parameter in the entire model** at R-hat 1.023 / ESS 110, its posterior slid
> toward the boundary (mean 0.032, 5.5 % quantile 0.0073), and it moved with the group-effect
> coord set (0.089 under one, 0.032 under another) — the "two fits disagree" signature again.
> The reason the data cannot pin it down: separating per-name *signal* dispersion from
> per-name *measurement* noise needs independent replication within a name, and the T=4
> lookback slices are strongly serially correlated (analyst targets are sticky), so they
> supply far less than 4 independent observations each. **The level itself is identified;
> its scale is not.** Fixing it deletes a stuck scalar from the sampler and leaves a plain
> regularised per-name effect, exactly as `GROUP_EFFECT_SCALE` does for the crossed
> intercepts. The 0.10 default brackets both learned estimates (0.032–0.089) with room above.

Three notebook-local flags sit alongside them:

- **`EXPORT_ARTIFACTS`** — `enable_artifact_export()` once, then `set_export_section(...)`
  at the top of every code cell below (CLAUDE.md's notebook convention; there is no
  enclosing `with` block per cell). Off by default, so an interactive run writes nothing
  to `KALMAN_PT_RESULTS_DIR`. `set_export_section` is a harmless no-op while disabled.
  It also fixes this run's **`run_id`** (§10c), which every exported frame is stamped with.
- **`WRITE_ANALYTICS`** — §10c's DB write. **Off by default**: it is a DROP-and-RECREATE of
  `analytics.kalman_filtered_price_targets`, the GEIB dashboard's only source. Use
  `scripts/export_kalman_analytics.py` for a production refresh (see the closing cell).
- **`ROBUST`** — Student-t vs Normal panel likelihood; `True` here, see §5.

In [ ]:
# --- notebook-local toggles ---------------------------------------------------
EXPORT_ARTIFACTS = True   # True -> persist figures/tables under KALMAN_PT_RESULTS_DIR
WRITE_ANALYTICS = True    # True -> §10c DROPs and RECREATEs the analytics table
ROBUST = True              # Student-t panel likelihood (matches validation + export)

# --- config overrides (uncomment as needed) ----------------------------------
# cfg = replace(cfg, panel_lookbacks=())                    # collapsed T=1 cross-section
# cfg = replace(cfg, state_innovation_scale=0.1)            # enable the opt-in AR(1) state
# cfg = replace(cfg, panel_response_extra=('pt_dispersion',))  # D=2, activates the ICM
# cfg = replace(cfg, enable_model_comparison=True)          # §9b (~3x sampling cost)
# cfg = replace(cfg, draws=500, tune=500, chains=2)         # smoke run

set_run_config(cfg)

if EXPORT_ARTIFACTS:
    enable_artifact_export()
    print('Artifact export ->', get_export_state().root)
else:
    print('Artifact export OFF (display only).')

_T = len(cfg.panel_lookbacks) + 1 if cfg.panel_lookbacks else 1
print(f'panel_lookbacks : {cfg.panel_lookbacks or "() — collapsed T=1 cross-section"}  (T={_T})')
print(f'state layer     : state_innovation_scale={cfg.state_innovation_scale} '
      f'({"AR(1) ON" if cfg.state_innovation_scale > 0 else "per-ISIN intercept only"})')
print(f'response_extra  : {cfg.panel_response_extra or "() — D=1, ICM dormant"}  '
      f'(supported: {tuple(KALMAN_PANEL_RESPONSE_EXTRA)})')
print(f'NUTS budget     : draws={cfg.draws} tune={cfg.tune} chains={cfg.chains} '
      f'cores={cfg.cores} target_accept={cfg.target_accept}')
print(f'likelihood      : robust={ROBUST} ({"Student-t" if ROBUST else "Normal"})   '
      f'write_analytics={WRITE_ANALYTICS}')

# Run identity. Every frame the SQL sink writes is stamped with this run_id, so a
# mixed-vintage analytics schema is one query away instead of a value-level diff.
_st = get_export_state()
print(f'run_id          : {_st.run_id}  (exported_at={_st.started_at:%Y-%m-%d %H:%M:%S} UTC)')

## §1 — Data load & feature-role resolution

- `load_kalman_df(engine, cfg)` — cross-sectional `pml.mv_pymc_kalman_pt` snapshot (one row
  per ISIN). The universe window (`next_earnings >= cfg.min_next_earnings`,
  `income_statement_report_date >= cfg.min_report_date`) is **config-driven** — roll the
  dates forward on `KalmanRunConfig`, never in the query.
- `load_feature_catalogue` — the `kalman_pt` rows of `pml.vw_pymc_feature_catalogue` (the
  SQL registry SSOT). Several `kalman_pt` roles are flipped via per-model overrides in
  `pml.pml_df_feature_alias`, so read the catalogue rather than assuming the base-row role.
- `resolve_feature_roles` — groups columns by `pymc_role` (catalogue SSOT, MV-schema fallback).

> **Not point-in-time.** `mv_pymc_kalman_pt`'s seven `days_*` horizons are computed against
> `CURRENT_DATE`, so a refresh on a different day silently shifts every one. Fine for the
> live screen; unusable as-is for a backtest. It is also part of why the whole `days_*`
> family is barred from the drift matrix (`KALMAN_TIME_COVARIATE_PREFIX`) and feeds the
> `t_scaled` axis instead.

In [ ]:
set_export_section('01_data')

kalman_df = load_kalman_df(engine, cfg)
feature_catalogue = load_feature_catalogue(engine)
roles = resolve_feature_roles(kalman_df, feature_catalogue)
print(f'{kalman_df.shape[0]:,} ISINs x {kalman_df.shape[1]} columns')
kalman_df.head()

## §2 — Exploratory data analysis

`run_eda` renders the EDA panels through a state-space lens: drift features → the
state-transition mean (`beta` slopes), noise wideners → the measurement-noise scale.
The panels carry decision context by default:

- the **industry ridge** is sorted by median with a 0% reference line;
- the **driver facets** annotate each facet with its Spearman ρ (plus an OLS trend line
  when `statsmodels` is available), so signal strength reads off directly;
- the per-coord group forests are **consolidated into one faceted panel**, gated to the
  coords the fused model actually uses as group effects, levels sorted by median with
  universe-median and 0% reference lines.

Note the EDA still shows the *excluded* families (`feat_pt_{median,high,low}_drift`, the
analyst composition legs) — they remain valid EDA / export columns and stay in the MV and
the catalogue. Only the **drift design matrix** drops them; see §3.

> **The size block became an earnings block.** The `feat_mcap_trend_1y` /
> `feat_mcap_vs_3yavg` / `feat_ev_vs_3yavg` panel is gone — those columns left
> `mv_pymc_kalman_pt` with the rest of the price-derived market-cap / EV family (§3), and
> they duplicated the momentum ladder already in the drift panels. In their place the EDA
> scans `feat_net_eps_drift`, `feat_last_{q,y}_surprise` and
> `feat_eps_beat_rate{,_annual}`, and the driver facets add EPS FY drift, beat frequency
> and last quarterly surprise. Read the facet ρ values with the coverage figures from §3 in
> mind: the two quarterly columns are ~47–54 % populated, so their facets are drawn on
> roughly half the universe.

In [ ]:
set_export_section('02_eda')

run_eda(kalman_df, roles)

## §3 — State-space feature mapping

`map_state_space_features` maps the catalogue's `kalman_pt` mutable_predictors onto Kalman
roles via `KalmanFilterPriceTarget.select_drift_features`, which applies the SSOT partition
in `KALMAN_DRIFT_EXCLUDED_FEATURES` (`KalmanFilterModel.py`): leakage, noise wideners, named
tilt drivers, support counters, rating counts, the collinear composition leg, Piotroski
components and `days_*` time covariates all stay out of the drift matrix.

### The drift matrix is 16 columns

`beta` was the **last failing convergence gate** on the full 6,540-name T=4 validation:
R-hat 1.026 / bulk-ESS 140 against 1.01 / 400, at **zero divergences**. Zero divergences
with slow mixing is the signature of poor conditioning, not bad geometry — and the design
carried condition number **1,580** with a smallest eigenvalue of 0.004. Two families each
restated one signal:

- **`KALMAN_PT_DRIFT_SIBLING_FEATURES`** — `feat_pt_{median,high,low}_drift` are the same
  `pml.target_drift()` run over the median / high / low target trails as `feat_pt_drift`
  runs over the mean. A consensus revision moves the whole band together: r = 0.81–0.89,
  VIF = 162 / 77 / 25 / 6.6.
- **`KALMAN_COLLINEAR_COMPOSITION_FEATURES`** — `feat_analyst_{bullish,bearish,neutral}_pct`
  and `feat_analyst_conviction` are all functions of the same six `num_*_ratings` buckets;
  `conviction` is literally `|bullish − bearish|` and the three pct legs sum to ~1.

One representative survives per family — **`feat_pt_drift`** and **`feat_analyst_rating`** —
taking the design from 21 columns / cond 1,580 / max VIF 162 to 15 / 23 / 3.8: 69× better
conditioning for a 0.6 % relative loss in explanatory power. An orthogonal replacement for
the dropped siblings (`high_drift − low_drift`, band-widening dynamics) was tried and
rejected: it correlates −0.003 with the response and moved R² by 0.0001.

### New in this build: the market-cap / EV family was swapped for an EPS family

Four columns left `mv_pymc_kalman_pt` entirely — `feat_mv_ev_drift`, `feat_mcap_trend_1y`,
`feat_mcap_vs_3yavg`, `feat_ev_vs_3yavg` (plus the eleven raw `market_cap_ev*` columns
`feat_mv_ev_drift` was built from). All four were **price-derived**: `market_cap` is
`last_price × shrs_out`, so they restated price history the design already carried through
`feat_price_drift`, `feat_price_chg_pct_3m`, `feat_one_day_return` and the
`feat_total_return_*` family — without saying anything about *why* analysts revise a target.

Their replacements are earnings-derived:

| new column                                         | what it is                                                            |
|----------------------------------------------------|-----------------------------------------------------------------------|
| `feat_net_eps_drift`                               | sign-preserving drift of the net-basic-EPS FY trail (`fy → neg5fy`)   |
| `feat_net_eps_drift_n`                             | valid-pair counter — **excluded** via `KALMAN_DRIFT_SUPPORT_COUNTERS` |
| `feat_last_q_surprise` / `feat_last_y_surprise`    | last realised EPS surprise, quarterly / annual                        |
| `feat_eps_beat_rate` / `feat_eps_beat_rate_annual` | beat frequency over the 5 quarterly / 6 annual surprise trail         |

**`pml.signed_drift`, not `pml.target_drift`.** Every pre-existing drift feature runs over a
strictly positive series (prices, targets, coverage counts, vol), where dividing by the raw
predecessor is fine. EPS crosses zero, and there the raw denominator **inverts** the signal:
a loss narrowing from −2.00 to −1.00 scores `(−1 − −2) / −2 = −0.5`, an improvement recorded
as negative drift. Winsorising caps the magnitude but keeps the wrong sign. The new helper
is identical except the denominator is `ABS(prev)`, so the same narrowing scores **+0.5**;
it agrees with `target_drift` exactly on positive series, and reuses `pml.target_drift_n`
for the min-points guard.

Measured on the live 6,538-name MV with one consistent response construction:

| set                         | k      | cond     | max VIF  | R²         |
|-----------------------------|--------|----------|----------|------------|
| mcap/EV family (previous)   | 15     | 24.7     | 4.36     | 0.6695     |
| **EPS family (this build)** | **16** | **19.7** | **4.25** | **0.6672** |

> **Read that honestly: this is not an R² play.** Explanatory power is flat (−0.3 %
> relative). The mcap/EV columns scored as well as they did *because* they restate price —
> and the response is analyst-implied upside off that same price, so their contribution was
> closer to duplication than to independent signal. What the swap buys is modestly better
> conditioning plus five predictors that are near-orthogonal to everything else: every new
> column lands at **VIF ≤ 1.19**, and the strongest correlation among them is
> `r(eps_beat_rate, eps_beat_rate_annual) = +0.369`.

**Coverage is the live caveat.** On the same 6,540-row MV: `feat_net_eps_drift` 100 %,
`feat_eps_beat_rate_annual` 85.2 %, `feat_last_y_surprise` 76.4 %, `feat_eps_beat_rate`
54.3 %, `feat_last_q_surprise` 46.6 %. The two quarterly columns are thin. That is tolerable
*here* because a sparse **predictor** zero-fills to the column mean after standardisation
(`prepare_kalman_panel_inputs`), which shrinks its `beta` toward 0 without dropping a single
name — unlike a sparse **response** series, which is what produced the rank-1 ICM
identification failure (max R-hat 4.45, min ESS 4.3) documented in `pml_feature_catalogue.sql`.
Watch the quarterly betas on the next full-scale fit; if they come back dead, drop them in
Python, not SQL.

> **v3's note here is retired.** v3 argued that `feat_analyst_conviction` was "clearly
> identified on the genuine T=4 panel (−0.024, 89% ETI excluding 0)" and should stay. It
> stayed null for the right reason: it never carried information the rest of the analyst
> family did not already have. `feat_one_day_return` — the *other* null beta in that note —
> is **kept**: its VIF is ~1, so it is merely uninformative rather than collinear.

> **Exclusions live in Python; removals live in SQL — they are different operations.**
> Flipping `pymc_role` to `'excluded'` in `pml_df_feature_alias` would drop a row from
> `vw_pymc_feature_catalogue` while `mv_pymc_kalman_pt` still emits the column, making
> `pml.assert_pymc_catalogue_coverage()` raise `MISSING_FROM_CATALOGUE`. The pruned
> sibling/analyst columns are therefore still emitted and still catalogued — only
> `KALMAN_DRIFT_EXCLUDED_FEATURES` bars them from the design. The mcap/EV four went the
> other way: they left the MV **and** the catalogue, so they need no exclusion entry. The
> cross-cutting trio survives in the other six `mv_pymc_*` views; only the `kalman_pt`
> `model_target` was `array_remove`d. `kalman_pt` reports 0 violations after the change.

In [ ]:
set_export_section('03_features')

drift_features, mapping = map_state_space_features(kalman_df, feature_catalogue)
print(f'{len(drift_features)} drift features (expected 16 on the current MV):')
for _f in drift_features:
    print('  -', _f)
mapping

## §4 — Fused-panel data containers

`prepare_kalman_panel_inputs` filters to log-space-usable rows and builds the
`KalmanPanelInputs`: the standardised `(isin, time, y_series)` response tensor `Y`, the
standardised time matrix `t_scaled`, the drift design matrix, the noise-widener drivers,
the systematic-risk / size / volume tilt inputs, and the categorical group coords.

**Time axis.** With `panel_lookbacks=('6m','3m','1m')` (the default) each lookback's implied
uplift `price_target_{lb}_ago / price_{lb}_ago − 1` is winsorised and `log1p`-mapped onto the
response scale — a **genuine** oldest→newest history panel (`T = len + 1`) with the snapshot
as the final step. Missing history cells (~1.5 %) are filled with the name's **own** snapshot
uplift, never a cross-sectional-mean fake observation. With `panel_lookbacks=()` the panel is
the collapsed T=1 cross-section and `t_scaled` is the standardised days-to-earnings covariate.

> **Primary response = log uplift.** `feat_log_uplift = log1p(feat_implied_upside)` —
> modelling the log keeps `expected_pt = last_price · exp(log_uplift)` strictly positive.
> `feat_implied_upside` itself is leakage-barred from the drift matrix.

### Two corrections that changed the exported numbers

1. **Support band (`UPLIFT_CLIP_LO` / `UPLIFT_CLIP_HI` = −0.95 / +5.0).** The response is
   winsorised to this decimal band *before* `log1p`, so the model never observes an uplift
   outside it. The two places that map back **out** of log space —
   `panel_posterior_upside`'s `expm1(latent)` and `summarize_panel_screen`'s `expm1(mc)` —
   were previously unbounded, and the 2026-08-10 export shipped `er_mean` up to 7.4e12 and
   `er_sd` up to 1.32e15 for ~1 % of names. Both directions now share one band, clipped in
   **log** space (`LOG_UPLIFT_CLIP_*`), which is sign-preserving and leaves `prob_pos`
   untouched. Read the result honestly: this truncates the posterior to the support the
   model was fit on, it does not extrapolate past it.
2. **Exact de-standardisation.** `KalmanPanelInputs` now carries the fit-time
   `response_mean` / `response_std`, so `_panel_response_stats` inverts the standardisation
   **by construction**. It previously recomputed the moments by tiling the *snapshot*
   column across `T` — correct only for the tile-based panel removed in 0.9.9.10. On the
   6,401-name T=4 run the pooled moments were `(0.207540, 0.249392)` vs the snapshot's
   `(0.224745, 0.241625)`, inflating `expected_upside` by **+2.32 pp** at a −0.5 latent
   through **+1.50 pp** at +1.0. A panel lacking the fields falls back to the legacy
   computation **with a warning**, never silently.

**Optional second response series.** `response_extra` promotes a key of
`KALMAN_PANEL_RESPONSE_EXTRA` to a second series (`D > 1`), activating the rank-1 ICM
(`mu_isin_loading` / `sigma_series`). The supplied series `pt_dispersion` =
`log1p(price_target_stddev_{lb} / price_{lb})` is a distinct signal (disagreement, not
direction) with a genuine `*_ago` trail. Promoting it **drops** the conflicting drift
predictor `feat_pt_noise_drift` (response ↔ predictor disjointness), with a printed note.

In [ ]:
set_export_section('04_panel')

panel = prepare_kalman_panel_inputs(
    kalman_df, roles, drift_features,
    history_lookbacks=cfg.panel_lookbacks,
    response_extra=cfg.panel_response_extra,
)
print('Y shape (isin, time, y_series):', panel.Y.shape)
print('response_names :', panel.response_names)
print('drift_names    :', panel.drift_names)
# Fit-time moments: these make the §10 de-standardisation exact by construction.
print('response_mean  :', getattr(panel, 'response_mean', None))
print('response_std   :', getattr(panel, 'response_std', None))
print(f'uplift support band: [{UPLIFT_CLIP_LO:+.0%}, {UPLIFT_CLIP_HI:+.0%}] (decimal), '
      'clipped in log space on the way back out')

## §5 — Build the fused panel model

`build_panel_model` wraps `build_fused_kalman_pt_model` and renders the model graph.

### Generative form

Per ISIN $i$, time step $t$, response series $d$ (primary $d{=}0$ is `feat_log_uplift`):

**Model A — risk-conditioned drift baseline.** A hierarchical regression on the standardised
drift design with crossed, fixed-scale sum-to-zero group intercepts:

$$\eta_i = X^{\text{drift}}_i\,\beta + \sum_g \big(e^{(g)}\big)_{[i]}
+ \underbrace{\sigma^{\text{lvl}}\, z^{\text{lvl}}_i}_{\text{only when } T>1},\qquad
\beta \sim \mathcal{N}(0,1),\ \ e^{(g)} \sim \text{ZSN}(\sigma{=}0.25)$$

$$\text{risk\_adj\_return}_i = \eta_i - \lambda\,z(\bar\beta_i)
- \gamma\,z(\text{mcap ratio}_i) - \kappa\,z(\text{rel\_volume}_i),\qquad
\text{achieve\_prob}_i = \sigma(\text{state\_now}_i)$$

**Model B — panel spine.**

$$\mu^{\text{reg}}_{i,t,d} = \alpha_{t,d} + W_d\,\text{state\_path}_{i,t},\qquad
y_{i,t,d} \sim \text{StudentT}\big(\nu,\ \mu^{\text{reg}}_{i,t,d},\
\sigma^{\text{isin}}_i \tau_d\big)$$

- $\alpha_{t,d}$ are **T direct per-time intercepts** — exactly identified. The former
  zero-anchored GRW deviations (+ a global slope) were mutually aliased per time slice on an
  isin-constant lookback axis and reproduced the historic scale×innovation ridge (2026-08-01
  T=4 run: 190 divergences, `alpha_level` R-hat 1.06 — 0 divergences after the fix).
- The per-series time slope $\beta^{(t)}_d$ is **not materialised at all** on an
  isin-constant axis. It was previously published as a Deterministic pinned at 0, which read
  as a fitted-and-dead parameter, gave arviz a constant to divide 0/0 on, and drew a flat
  zero line in the §13b slope panel. It returns unchanged when `t_scaled` genuinely varies.
- $W_d$ is the sign-fixed coregion loading (primary ≡ 1); $\tau_d$ the per-series noise
  diagonal; $\nu \ge 2.5$ the tail dof. Both are inert while `D == 1`.
- $\text{state\_path}_{i,t} = \mu^{\text{isin}}_i$ with the AR layer off, and
  $\text{state\_now}_i = \text{state\_path}_{i,-1}$.

**Crossed group effects** are `_FUSED_KALMAN_GROUP_EFFECTS = ('trading_region', 'sector',
'style_class', 'size_class')` at a **fixed** `GROUP_EFFECT_SCALE = 0.25` — the group SD is
not learned (structurally non-identified on one slice). `region` was **dropped**: it and
`trading_region` agree for 96.12 % of the universe (Cramér's V 0.938, only cross-listings
differ) against V ≤ 0.24 for every other pair, and once the per-ISIN intercept landed they
became the two worst-mixing globals. `trading_region` is kept because listing venue
determines analyst coverage and currency — the mechanism the drift features measure.

> **The `robust` trap.** `robust=True` (Student-t) is `build_fused_kalman_pt_model`'s own
> default, and it is what `scripts/validate_kalman_state.py` and
> `scripts/export_kalman_analytics.py` fit. But `pymc_kalman_filter_pt.main()` defaults to
> **`robust=False`** (Normal). Exporting on the Normal variant ships numbers no gate has
> seen, so this notebook sets `ROBUST = True` in §0b — pass `robust=True` explicitly if you
> ever call `main()` directly.

In [ ]:
set_export_section('04_panel')

VOLUME_PENALTY = 0.25  # HalfNormal prior scale for the learned volume_loading tilt; 0.0 disables
model = build_panel_model(panel, robust=ROBUST, volume_penalty=VOLUME_PENALTY, config=cfg)
print('state layer vars:',
      [v for v in ('state_path', 'state_now', 'sigma_isin_level', 'sigma_state')
       if v in model.named_vars])
pm.model_to_graphviz(model)

## §6 — Prior predictive checks

`run_prior_predictive` draws `cfg.prior_draws` prior samples of `expected_return`, the
decision latent (resolved through `resolve_screen_latent`, i.e. `state_now`), `achieve_prob`
and `sigma_isin`, then de-standardises onto the interpretable **percent** implied-upside
scale and compares against the empirical distribution — the Bayesian-workflow stage
contract, not a density doodle.

`state_path` is deliberately **skipped**: an `(isin, time)` tensor over prior draws is large
and carries no extra prior information beyond `state_now`.

In [ ]:
set_export_section('06_prior')

prior_idata = run_prior_predictive(model, panel, cfg)
print('decision latent:', KALMAN_SCREEN_LATENT,
      '->', resolve_screen_latent(prior_idata.prior).name)
prior_idata.prior

## §7 — Posterior inference (NUTS)

`sample_posterior` tries `nutpie → numpyro → pymc` in priority order (the same order
`sample_with_fallback` uses for the §9b arms and the validation script) and merges the prior
groups into the posterior `DataTree`. The budget comes from the config:
`draws/tune/chains/target_accept/random_seed`. Passing `panel=` stamps the drift-feature
aliases plus their catalogue metadata onto `constant_data['drift_features']` via
`stamp_feature_provenance`.

> **`cores=1` in the IDE kernel.** Launching nutpie's parallel native workers inside an
> IDE-managed Jupyter kernel on Windows can crash the kernel process — an uncatchable native
> crash that surfaces only as *"Connection to IDE-Managed Server is lost"*. Chains run
> sequentially here; the standalone script path uses `cfg.cores`.

> **Sampler choice is not a micro-optimisation.** The project forces the PyTensor
> pure-Python VM, under which PyMC's own NUTS produced **zero draws in 42 minutes of CPU**
> on the local-level panel (~16.8k extra parameters on a 5.6k-ISIN T=4 panel). Anything
> sampling this model must go through `sample_posterior` / `sample_with_fallback`, never
> bare `build_sample_kwargs` (whose `nuts_sampler=None` default lands on that path).

**Validation history** (why the settings look like this):

| run                                    | geometry                             | result                                              |
|----------------------------------------|--------------------------------------|-----------------------------------------------------|
| T=1 baseline                           | direct intercepts, no time axis      | 0 div, R-hat ≤ 1.01, ESS > 400                      |
| T=4 (2026-07-31/08-01, GRW deviations) | aliased level/slope/innovation block | 315 / 190 divergences, `alpha_level` R-hat 1.06     |
| T=4 reparameterised (2026-08-01)       | per-time direct intercepts           | 0 div, worst R-hat 1.00, ESS ≈ 1.6k, 15.7 min       |
| T=4 + per-ISIN latent, 21 drift cols   | `draws=1000`                         | 0 div, but `beta` R-hat 1.026 / ESS 140             |
| T=4, 15 drift cols, `region` dropped   | `draws=1000`                         | 0 div, `beta` R-hat ≤ 1.0121 / ESS 236–296          |
| **T=4, 15 cols, `draws=2000`**         | **shipped default**                  | **0 div, max R-hat 1.0090, min ESS 678.3, ≈32 min** |

`log_likelihood` is hard-coded **off** here (it roughly doubles idata size and no production
path consumes it), so `az.loo` / `az.compare` raise on this object. Attach it post-hoc with
`attach_log_likelihood(idata, model)` — the `idata_kwargs={'log_likelihood': True}` route is
silently stripped under nutpie. §9b does exactly that.

In [ ]:
set_export_section('07_posterior')

idata = sample_posterior(model, prior_idata, cores=1, panel=panel, config=cfg)
print('Group effects fitted:', present_group_effects(idata))
print('Divergences:', int(idata.sample_stats['diverging'].sum()))
idata.posterior

## §8 — Posterior predictive checks

Calibration of the standardised `(isin, time, y_series)` likelihood:

- **Pooled ECDF overlay** — replicate ECDFs vs the observed ECDF.
- **t-stat calibration** (`mean`, `std`) — observed T(y) inside the replicated distribution.
- **94 % coverage per `y_series`** — printed and charted against the 0.94 target line.
- **94 % coverage per time step** — *new in 0.9.9.14*, and the statistic that tests the
  state layer specifically. Pooled coverage can look correct while the model is
  over-confident at the oldest lookbacks and over-dispersed at the snapshot; a **monotone
  drift across `t`** indicates a mis-set innovation scale. This is what falsified the
  literal cumulative random walk: its marginal variance grows as √t and the full-scale run
  produced 89.9 % → 95.3 % → 97.2 % → 98.2 % against a 94 % target. The shipped build
  measured 91.6–92.5 % across `t` — flat, no drift.
- **PIT ECDF** — uniform / in-band when calibrated.

> ### ⚠️ Three of these currently FAIL (2026-08-15 run, n = 25,948)
>
> Coverage passing is **not** evidence of calibration here — it is what an over-wide
> interval produces. Read the coverage bar together with the two t-statistics:
>
> | check | observed | replicated | verdict |
> |---|---|---|---|
> | 94 % PI coverage | 92.0–92.4 %, flat across `t` | target 94 % | near |
> | PIT ECDF uniformity | — | max deviation −0.027 @ 0.28, +0.021 @ 0.71 | **p = 0.00** |
> | T = mean | 0.000 | −0.040, range ≈ [−0.065, −0.015] | **outside** |
> | T = std | 1.03 | 1.40–1.60, tail to 9+ | **outside** |
>
> All three say the same thing from different angles: the predictive is **biased low in
> location and ~40–50 % over-dispersed in scale**. The PIT curve is a clean S — too little
> mass in the lower quantiles, too much through the middle and upper.
>
> **Root cause is the response support band, not the tail parameter.** `nu` sits pinned at
> its floor (`nu = 2.5 + nu_tail`, with `nu_tail` at 0.0299, ETI89 [0.0059, 0.067]), because
> the `[−0.95, +5.00]` winsorisation admits +500 % implied-upside observations as real data
> and the Student-t tail is the only thing that can absorb them. Widening the tail
> over-disperses the *whole* predictive rather than isolating the outliers.
>
> **Do not relax the `nu ≥ 2.5` floor** (`KalmanFilterModel.py:3085-3100`): it exists to
> remove an improper corner of the joint density where the sampler previously slid to
> `nu → 0.06`, `sigma_base → 0`, the mean structure switched off and every ISIN collapsed
> onto a flat ≈30 % screen. The fix belongs on the response side — tighten the fit-time
> clip (Phase B), or move to a two-component mixture likelihood.

In [ ]:
set_export_section('08_ppc')

run_posterior_predictive(model, idata, panel)

## §9 — MCMC diagnostics

`run_diagnostics` reports R-hat / bulk-&-tail ESS, divergences, trace / rank-dist / forest
views, the NUTS energy, prior→posterior contraction, the ESS evolution and the variance
partition (printed **and** rendered as a stacked share bar). Gates follow Vehtari et al.
(2021): **R-hat < 1.01**, **ESS > `MIN_ESS_GATE` = 400**.

The variable groupings are the module's SSOT:

- **`FUSED_SCALAR_VARS`** — `sigma_base`, `nu`, `sigma_state`, plus the learned sign-fixed
  `risk_loading` / `size_loading` / `volume_loading`. Absent vars are skipped, so
  `sigma_state` appears only when the AR(1) layer is enabled on a `T > 1` panel. **Watch it:
  a posterior pressed against 0 means the panel carries no per-name time dynamics and the
  state layer is dead weight.**
- **`FUSED_VECTOR_VARS`** — `beta`, `alpha_level`, `beta_slope`, `mu_isin_loading`,
  `sigma_series`. `beta_slope` exists only when `t_scaled` varies across ISINs;
  `mu_isin_loading` / `sigma_series` only when `D > 1`.
- Per-coord `sigma_<coord>` are appended at runtime from the coords actually present.

`sigma_alpha_innov` / `sigma_beta_innov` no longer exist (the per-time direct intercepts
removed them). Constant posterior variables are filtered through `_degenerate_posterior_vars`
**before** the `azs.rhat` / `azs.ess` sweep, so the `invalid value encountered in scalar
divide` warning is gone at source rather than suppressed.

### Reading the shipped run (`run_id = 8538a5d6ee25`, 2026-08-16)

Sampling is clean and the budget question is closed: **0 divergences, max R-hat 1.0022,
min bulk ESS 784, min tail ESS 1,274 — none of the 52 summarised parameters under the 400
gate**, with E-BFMI 0.91–1.0 across four chains and marginal/transition energy densities
essentially overlapping. No funnel pathology.

What the numbers *do* say is structural rather than computational:

| parameter | mean | ESS bulk | reading |
|---|---|---|---|
| `beta[feat_pt_accuracy_1y]` | **−1.3074** | **784** | lowest ESS in the model |
| `beta[feat_pt_achievement_1y]` | **−1.1233** | **837** | |
| `beta[feat_pt_drift]` | **+1.2933** | **845** | |
| `sigma_base` | 0.1484 | 1,085 | ~66 % of total posterior-mean scale |
| `nu` | 2.5259 | 8,359 | pinned at the 2.5 floor — see §8 |
| `risk_loading` | 0.0018 | 5,026 | on its lower bound |
| `size_loading` | 0.0018 | 5,985 | on its lower bound |
| `volume_loading` | 0.0010 | 5,994 | on its lower bound |

- **Three features carry the model.** Every other coefficient sits below 0.15 in absolute
  value, and those same three hold the three lowest ESS values (784–845 against
  2,185–10,622 for the rest) — the signature of residual correlation inside the
  price-target-history family, which §3's pruning deliberately kept whole. Phase C
  revisits it.
- **All three learned tilts are switched off by the data**, each collapsed onto the lower
  bound of its positive support. `VOLUME_PENALTY` in §5 is tuning a term the posterior has
  already zeroed.
- **Two-thirds of the modelled scale is unstructured** — trading region ≈13 %, sector
  ≈13 %, style ≈4 %, size ≈3 %, and `sigma_base` the rest.

> **`sigma_isin_level` is a CONSTANT — do not read it as a fitted quantity.** It reports
> mean 0.1, **sd exactly 0, ESS == total draws, R-hat NaN, mcse 1.6e-19**, because it is
> `pm.Deterministic(pt.constant(isin_level_scale))`. It cannot "collapse toward 0" and a
> gate on its posterior would be meaningless — `scripts/validate_kalman_state.py` gates on
> **`z_isin_level` being present** and on the realised effect sd instead (0.0454 on this
> run, 45 % of the fixed scale). See §0b for why the scale is fixed.
>
> Two scales are **missing from the variance-partition chart**, which shows 5 of 7:
> `sigma_isin_level` (above) and `sigma_time_free`. The latter is worth looking at directly
> — it came back `[3.25, 2.73, 1.85]` against an anchor of 1.0 at t=3, i.e. the older
> lookbacks carry two to three times the noise scale. That is the single most informative
> number about how much the T=4 panel is actually buying, and it currently appears nowhere
> in the exported diagnostics.

In [ ]:
set_export_section('09_diagnostics')

run_diagnostics(idata, panel)

## §9b — Model comparison (ELPD / LOO) — opt-in

`run_model_comparison` closes the last `❌` in this module's Bayesian-workflow coverage row.
The two arms differ in exactly one thing — whether the per-ISIN latent may evolve over the
panel:

- **`local_level`** — `state_innovation_scale` from the config (use `0.1`);
- **`static`** — `state_innovation_scale=0.0`, pinning the state at its t=0 anchor, i.e. the
  pre-0.9.9.14 time-constant build.

Both are refit on the same subsampled panel so the ELPD contrast is like-for-like.

**Why it is opt-in:** each arm needs a pointwise `log_likelihood` group of
`chains × draws × n_isin × T × D` floats — **~820 MB per arm** at full panel size — and both
arms are refit, so this roughly **triples** the run's sampling cost. `comparison_max_isins`
(default 800) bounds the ISIN axis and the retained fraction is printed, so a truncated
comparison never reads as a full one. Needs `T > 1`; it prints a skip on a T=1 panel.

**Reading it:**

- The group is attached post-hoc with `attach_log_likelihood` (`pm.compute_log_likelihood`).
  The `idata_kwargs={'log_likelihood': True}` route does **not** work — nutpie ignores
  `idata_kwargs` and `build_sample_kwargs` strips it.
- ArviZ 1.x exposes the value as **`.elpd`**. `.elpd_loo` was removed and a `getattr`
  fallback on the old name silently yields `nan`, even though the `ELPDData` repr still
  prints the `elpd_loo` row label.
- Judge on **`elpd_diff` vs `dse`**, not on Pareto k-hat. High k-hat on the state arm is
  expected by construction (it carries a per-ISIN latent path) — the documented weakness of
  PSIS-LOO for models with per-observation latents. A margin inside ~2 `dse` is
  inconclusive, not a win.

In [ ]:
set_export_section('09b_comparison')

# OPT-IN: refits BOTH arms and computes a pointwise log_likelihood for each
# (~820 MB per arm at full panel size) -> roughly 3x this run's sampling cost.
# Uncomment to run.
#
# cmp_df = run_model_comparison(
#     panel,
#     config=replace(cfg, enable_model_comparison=True, state_innovation_scale=0.1),
#     robust=ROBUST, volume_penalty=VOLUME_PENALTY,
# )
# cmp_df

print('§9b skipped (opt-in). Uncomment the call above, or set '
      'cfg = replace(cfg, enable_model_comparison=True) before kf.main(...).')

## §10 — Expected price targets: posterior screen

`summarize_panel_screen` → a `ScreenContext` with the posterior `expected_upside` /
`expected_pt` draws, the per-ISIN screening `results` table and the structural-TS
Monte-Carlo summary. The `er_*` columns are genuine **decimal returns** (0.25 = +25 %);
percent scaling happens only at display boundaries. The Monte-Carlo horizon / damping come
from `cfg.mc_horizon` / `cfg.mc_rho`.

> **The decision latent is `state_now`.** Since the state layer landed, every consumer
> resolves the per-ISIN quantity through `resolve_screen_latent` — the **filtered level at
> the final (snapshot) time step**. The posterior *variable* `risk_adj_return` is now only
> the t=0 structural anchor. The screen and export **column** named `risk_adj_return` keeps
> its name and units but reports the filtered level; the two coincide exactly when `T == 1`
> or the state is pinned off, so the fallback is not a degraded path.

> **De-standardisation is exact, and `expm1` is bounded.** The inverse uses the panel's
> fit-time `response_mean` / `response_std` (§4), removing the +1.5–2.3 pp overstatement,
> and the log-space draws are clipped to `[LOG_UPLIFT_CLIP_LO, LOG_UPLIFT_CLIP_HI]` before
> `expm1`. Clipping in log space is sign-preserving, so `prob_pos` is untouched. Names whose
> distribution pins at the cap are flagged in §10c rather than published with a fabricated
> ranking score.

Renders: the per-industry posterior forest (0-line, sorted), the fused-model internals panel,
and the comparative-returns views (percent-space shrinkage scatter, distributional KDE
overlay, per-sector forest).

In [ ]:
set_export_section('10_screen')

screen = summarize_panel_screen(idata, panel, horizon=cfg.mc_horizon, rho=cfg.mc_rho)
results = screen.results
results.head(15)

### §10b — CVaR-aware risk analytics & sizing (RiskBook)

Single source of truth for the risk layer (`RiskBookModel.compute_cvar_aware_book`): a
per-name tail statistic on the posterior upside draws, a reward-to-tail-risk (STARR)
ranking, and a per-name-capped long book with joint-draw portfolio aggregates. Knobs
resolved from the config: `cvar_alpha` / `weight_cap` / `k_book` / `p_long` /
`mcap_country_r_max`. The same `RiskBook` feeds the §10c export and the §14b
recommendations.

All columns are raw decimals, including `cvar05` and `exp_vol`.

> ### These columns measure ESTIMATION uncertainty, not return risk
>
> `cvar05`, `exp_vol` and everything derived from them come from the **posterior
> expected-upside draws**. That is a statement about how well the model knows each name's
> upside, not about how much the stock can move. Measured on the shipped run
> (`run_id = 8538a5d6ee25`):
>
> | column | what it is NOT | what it is |
> |---|---|---|
> | `cvar05` | a loss | the conditional **mean** of the worst 5 % of the upside draws. Positive for **5,475 of 6,487** names (universe mean **+0.185**). A value of 0.48 is a **+48 % return**, not a 48 % drawdown. |
> | `exp_vol` | a return volatility | the posterior sd of those draws — universe mean **0.0253 (2.5 %)**, roughly an order of magnitude below any realised equity vol. |
> | `expected_sharpe` | an investment Sharpe ratio | `er_mean / er_sd` over MC draws of the **log price-target uplift**. A t-statistic on the uplift estimate: universe median **1.56**, book names 5–7. |
>
> The genuine dispersion is `tail_risk = max(expected_upside − cvar05, −er_p05, 0.01)`, and
> `starr = expected_upside / tail_risk` is what the book actually ranks on. For a downside
> quantile of the *forward-return* distribution, use `er_p05`.
>
> `RiskBookModel.py` already carried this caveat for `ret_vol_ratio` ("internal only, NOT a
> Sharpe ratio — the denominator is parameter uncertainty"); as of 2026-08-16 the same
> register is applied to the exported siblings, in the `COMMENT ON COLUMN` text and in the
> GEIB labels. **No column was renamed** — the dashboard contract in
> `dashboards/geib/data.py` is unchanged.

> **`prob_pos` in this frame is degenerate — but it never reaches the dashboard.**
> `results['prob_pos']` (posterior P(upside > 0)) is pinned at exactly 1.0 for **4,960 of
> 6,487** names, so its 25th percentile and median are both 1.0 and it cannot order a
> ranking. It lives here and in `analytics."10_screen_results"` /
> `"10b_risk_analytics"` only. The analytics export carries `mc_prob_pos` (Monte-Carlo,
> mean 0.864 / median 0.968, 2 pinned) and `p_upside_pos_cond` (mean 0.457, none pinned) —
> both discriminating. Rank on those, or on `er_p05`; never on `prob_pos`.

> **Finite guard (2026-08-16).** All three ratios now floor their denominator at
> `MIN_RATIO_DENOMINATOR = 1e-4` and re-check `np.isfinite`. The previous `> 0` test was
> passed by a denormal `er_sd` of ~4e−16 — which is how `expected_sharpe_ratio = −4.28e15`
> reached the dashboard for clip-pinned names. Post-fix the exported column spans
> **−33.73 … 7.998**; real values are untouched (Tencent 7.4928 before and after).

**Book composition is driven by the eligibility gate, not by STARR.** `mcap_country_r_max`
admits only ~813 of 6,487 names before ranking, which is why the 25-name book is heavily
China + United States by weight. The `div` figure in `summary` measures weight spread,
**not** country exposure — do not read it as the latter.

In [ ]:
set_export_section('10b_risk')

risk_book = compute_cvar_aware_book(idata, panel, screen, results, config=cfg)
print({k: round(v, 4) if isinstance(v, float) else v
       for k, v in risk_book.summary.items()})
risk_book.book.head(25)

### §10c — Analytics export & decision dashboard

`export_analytics` maps the posterior onto `analytics.kalman_filtered_price_targets`
(raw-decimal convention, **102 columns** — 100 plus the two provenance columns below). The
overview dashboard carries the portfolio star (aggregate E[r] / vol / CVaR from
`RiskBook.summary`), the held-name efficient hull on the risk-return map, the MC return fan
over the sized book, and the shrinkage view on signed-log axes.

#### The `out_of_support` flag — now tested on BOTH clip bounds

A name whose forward-return distribution is pinned at a winsorisation bound has no reliable
ranking metric: its `er_sd` collapses toward 0, which makes
`expected_sharpe_ratio = er_mean / er_sd` explode. That is the dangerous direction of
failure — an unbounded 1e15 gets noticed, but a Sharpe of 717 *sorts straight to the top of
any risk-adjusted screen* while marking precisely the names the model understands **least**.
So for those rows `expected_sharpe_ratio`, `reward_to_cvar` and `cvar_book_weight` go NULL
(weight re-filled to 0) and `out_of_support = TRUE`; identity, price targets and the raw
`er_*` distribution are retained, so nothing vanishes silently.

The clip is two-sided (`UPLIFT_CLIP_LO, UPLIFT_CLIP_HI = −0.95, 5.0`) and the detector now
tests both ends, using the percentile that reaches the bound first in each direction:

| direction | test | why that percentile |
|---|---|---|
| +500 % cap | `er_p05 >= UPLIFT_CLIP_HI − 1e-6` | `er_mean` averages the clipped draws, so a handful landing below the cap drag it ~1e-4 under — a first attempt on `er_mean` matched **zero** of the 18 affected names. `er_p05` lands exactly on the cap once ~95 % of draws are clipped. |
| −95 % floor | `er_p95 <= UPLIFT_CLIP_LO + 1e-6` | the mirror image: a distribution is pinned at the floor exactly when its 95th percentile has reached it. |

> **The floor test was missing until 2026-08-16, and it is the side that fires.** On the
> 2026-08-15 export the cap test matched **0 of 6,487** rows, while several names sat with
> every draw at −95 %, `er_sd` ≈ 0 and `expected_sharpe_ratio = −4.28e15`, poisoning every
> `AVG` and `ORDER BY` the dashboard ran over that column.
>
> On the shipped run the symmetric test flags **4 names, all at the floor, none at the cap**:
> Kioxia (285A), Yuanjie Semiconductor (688498), AXT (AXTI) and Sandisk (SNDK).
>
> **Test the percentile, not the standard deviation.** Yuanjie has `er_sd = 0.0020`, not
> zero — a guard keyed on `er_sd == 0` would have missed it. `er_p95 = −0.95` catches it.

#### Provenance: `run_id` / `exported_at`

Every frame the SQL sink writes — the seven curated `_SQL_EXPORT_ARTIFACTS` bulk frames and
`kalman_filtered_price_targets` — is stamped by `stamp_export_provenance()` with this run's
`run_id` and `exported_at` before its DDL is rendered. `check_export_vintage()` reports the
run id per table and warns when they differ; it runs automatically after a `write=True`
export.

> **The mixed-vintage schema had a structural cause, not a one-off one.** The original
> diagnosis — "the 2026-08-15 run happened to skip five frames" — was wrong.
> `export_all_artifacts()` is what writes `04_panel_frame`, `10_screen_results`,
> `10_screen_mc_summary`, `10b_risk_analytics` and `10b_risk_book`, and
> `scripts/export_kalman_analytics.py` **never called it**. The production path therefore
> wrote **2 of 7 curated tables by construction** — this table plus
> `09_diagnostics_01_table`, which `run_diagnostics` exports inline — so the divergence
> reappeared on *every* refresh. Cross-table `er_mean` disagreement was **6,425 of 6,427**.
> The script now calls `export_all_artifacts` (omitting `prior_idata` / `idata` so it does
> not re-dump the ~8 GB posterior DataTree), and the disagreement is **0 of 6,487**.

> **Beware speculative `SELECT` on PostgreSQL.** The first release of
> `check_export_vintage()` probed each table with `SELECT run_id` and caught the error. An
> `UndefinedColumn` **aborts the surrounding transaction**, so the first unstamped table
> poisoned the connection and every later query failed with `InFailedSqlTransaction` — it
> reported all eight tables unstamped, *including the two that were correctly stamped*.
> Resolve column existence from `information_schema` first.

> **0.9.9.14 changes the exported VALUES, not the schema.** The de-standardisation fix and
> the per-ISIN latent both move numbers; the raw-decimal convention is unchanged, and the
> 2026-08-16 provenance columns are additive, so **no destructive DDL migration is needed**
> — but the export and the GEIB dashboard deploy still ship as a pair.

`WRITE_ANALYTICS` gates the DB write: it is a DROP-and-RECREATE of the dashboard's only
source. For a production refresh, run `scripts/validate_kalman_state.py` (gates 1–3 must
pass), then `scripts/export_kalman_analytics.py` — see the closing cell.

In [ ]:
set_export_section('10c_analytics')

kalman_results = export_analytics(idata, panel, screen, risk_book=risk_book,
                                  write=WRITE_ANALYTICS)

# Out-of-support: report BOTH clip directions. The floor side is the one that
# fired unflagged before 2026-08-16.
if 'out_of_support' in kalman_results.columns:
    _oos = kalman_results['out_of_support'].fillna(False)
    _at_cap = (kalman_results['er_p05'] >= UPLIFT_CLIP_HI - 1e-6).fillna(False)
    _at_flr = (kalman_results['er_p95'] <= UPLIFT_CLIP_LO + 1e-6).fillna(False)
    print(f'out_of_support: {int(_oos.sum())} of {len(kalman_results)} names '
          f'({int(_at_cap.sum())} at the +{UPLIFT_CLIP_HI:.0%} cap, '
          f'{int(_at_flr.sum())} at the {UPLIFT_CLIP_LO:.0%} floor) '
          f'— ranking metrics NULLed, er_* retained')
    if int(_oos.sum()):
        display(kalman_results.loc[_oos, ['isin', 'ticker', 'name', 'er_mean',
                                          'er_sd', 'er_p05', 'er_p95']])

# The finite guard should leave nothing pathological in the ranking columns.
for _c in ('expected_sharpe_ratio', 'reward_to_cvar'):
    _v = kalman_results[_c].dropna()
    if len(_v):
        print(f'{_c:>22}: min={_v.min():+.4g}  max={_v.max():+.4g}  '
              f'|v|>100: {int((_v.abs() > 100).sum())}  (expect 0)')

# The OTHER six curated frames. export_analytics writes only THIS table; without
# the call below a notebook run leaves 04_panel_frame / 10_screen_results /
# 10_screen_mc_summary / 10b_risk_analytics / 10b_risk_book on whatever fit last
# touched them — the exact mixed-vintage schema found on 2026-08-16. Omitting
# prior_idata / idata keeps export_all_artifacts from re-dumping the ~8 GB
# posterior DataTree; missing keys are skipped by design.
if WRITE_ANALYTICS:
    kf.export_all_artifacts({
        'panel': panel,
        'results': results,
        'screen': screen,
        'risk_book': risk_book,
        'kalman_results': kalman_results,
    })
    kf.check_export_vintage()   # expect: one run_id across every existing table

kalman_results.head()

### §10K — Universe-consensus fit

`run_universe_kalman_fit(kalman_df)` is the one-call driver `main()` uses: it pools every
row's `price_target*_ago` trail into a weekly-median consensus series
(`build_universe_consensus`), routes it through the canonical `fit_kalman_model` — the
funnel-free marginalized GRW (+ trend), nutpie, spot-anchored at the universe-median
`last_price` — structurally forecasts to the universe-median fiscal events, and calls
`report_universe_kalman_fit` internally.

v3 hand-assembled these three steps in the notebook; the driver is the SSOT, so this cell is
one call. Override any fit kwarg by keyword (e.g. `samples=`, `chains=`, `trend=`).

The structural forecast renders in **return space** — observed vs expected returns per
fiscal event with a 0 % break-even line — and the forecast table always carries
`implied_upside_pct`.

In [ ]:
set_export_section('10k_universe')

universe_fit = run_universe_kalman_fit(kalman_df, random_seed=cfg.random_seed)
universe_fit

## §11 — Single-ISIN time-series Kalman filter (+ §11b stochastic volatility)

The literal single-security GRW filter on the richest `*_ago` history (time axis anchored on
`income_statement_report_date`). The candidate pull is config-driven
(`min_mcap_country_rank`, `candidate_limit`). The structural forecast is the **return-space**
panel (`plot_kalman_forecast_returns`): observed implied returns, the smoothed
implied-upside band, nested latent/predictive forecast bands per fiscal event (the gap
between them is the analyst observation noise), per-horizon +X % annotations.

§11b refits with stochastic volatility — its σ_obs(t) path renders as the companion row
(posterior median).

In [ ]:
set_export_section('11_single_isin')
single_ctx = run_single_isin_filter(panel.frame, engine, cfg)

set_export_section('11b_single_sv')
run_single_isin_stochastic_vol(single_ctx)

## §12 — Mingled-ISIN earnings-window cohort filter (+ §12b stochastic volatility)

Every ISIN whose `next_earnings` lands within ±`cfg.earnings_window_days` of today is
unpivoted and the cross-sectional **median** target taken per shared as-of date — one
earnings-cohort consensus series, fit with the marginalized GRW (+ trend). Return-space
forecast + upside-bearing forecast table, as in §11.

In [ ]:
set_export_section('12_mingled')
mingled_ctx = run_mingled_cohort_filter(panel.frame, engine, cfg)

set_export_section('12b_mingled_sv')
run_mingled_cohort_stochastic_vol(panel.frame, mingled_ctx)

## §13 — Granular earnings-cohort posterior-predictive forest (+ §13.1 further views)

Keeps the §12 cohort definition but stays per-ISIN granular, reusing the fitted fused
posterior (no refit): per-name `expected_pt` posterior forests with the raw analyst targets
overlaid and pooled reference bands.

> **§13b panel (d) changed.** It now plots the **`state_path` median with a 10–90 %
> cross-sectional band** instead of the `beta_t` slope, falling back to `beta_t` only on a
> genuinely isin-varying time axis. Read it directly: a visibly **widening** band from t=0 is
> the state layer working; a **flat** one means `sigma_state` collapsed and the AR layer is
> dead weight. With the AR layer off (the default) the path is constant across `t` by
> construction — `state_path` equals `mu_isin` — so a flat band there is expected, not a
> failure.

In [ ]:
set_export_section('13_forest')
forest_ctx = run_granular_forest(idata, results, panel, screen, engine, cfg)

set_export_section('13b_further_views')
run_granular_further_views(prior_idata, panel, screen, forest_ctx)

## §14 — Comprehensive summary & actionable recommendations

`run_summary` consolidates the run into an earnings-cohort vs baseline vs universe read; the
cross-sectional table and the sector mix render as decision panels (grouped metric bars;
cohort-vs-universe sector-tilt diverging bar).

`run_recommendations` (§14b) turns the posterior into risk-aware signals. The
group-allocation block renders the **shrunk-excess forest** (per-coord OW/UW bands,
verdict-coloured); the CVaR sizing block renders the **book composition** chart with the
portfolio aggregates (E[upside] / CVaR5 / reward-to-CVaR / diversification).

In [ ]:
set_export_section('14_summary')
run_summary(results, screen, forest_ctx, mingled_ctx)

set_export_section('14b_recommendations')
run_recommendations(idata, panel, results, screen, forest_ctx, risk_book=risk_book)

### §14.1 — Screen overview, risk/return screen & top-candidate forest

These plotting functions live in the module — import, don't re-define:

- `plot_screen_overview` — upside distribution + top-N ranked names with HDIs;
- `plot_risk_return_scatter` — interactive upside vs posterior-uncertainty screen
  (colour = sector, size = market cap);
- `plot_top_candidate_forest` — posterior expected-upside forest of the top names.

In [ ]:
set_export_section('14_summary')

plot_screen_overview(results, top_n=50)
plot_risk_return_scatter(results)
plot_top_candidate_forest(screen, results, top_n=50)

---
### Artifacts available at the top level

`cfg`, `kalman_df`, `roles`, `drift_features`, `panel`, `model`, `prior_idata`, `idata`,
`screen`, `results`, `risk_book`, `kalman_results`, `universe_fit`, `single_ctx`,
`mingled_ctx`, `forest_ctx`.

### One-shot alternatives

```python
# Full workflow in one call. NOTE main() defaults to robust=False (Normal likelihood)
# while the builder, the validation script and the export script all use Student-t.
kf.main(run_eda_section=True,write_analytics=False,robust=True,export_results=True,config=cfg)
# -> {'idata', 'prior_idata', 'results', 'kalman_results',
#     'panel', 'screen', 'risk_book', 'universe_fit'}
```

```powershell
. .\set_env.ps1

# Gate the re-export. Six gate groups:
#   1 convergence (divergences, R-hat, ESS)
#   2 the per-ISIN latent is alive (z_isin_level present, realised effect sd)
#   3 the predicted sigma_base-falls / per-name-sd-widens signature vs the
#     no-latent baseline  [needs the second fit; --no-static skips it]
#   4 per-time PPC coverage
#   5 the de-standardisation delta (no fit required)
#   6 the EXPORTED table (2026-08-16): no non-finite or |v|>100 ranking metric,
#     every clip-pinned row flagged out_of_support with NULL ranking columns,
#     one run_id across all seven tables.
#     ADVISORY while the live table carries no run_id (it then predates the
#     fixes, and gating a new fit on the old export's quality is circular);
#     HARD once the table is stamped.
python scripts\validate_kalman_state.py

# Production refresh (robust=True, stops after §10c). Since 2026-08-16 this also
# calls export_all_artifacts, so it writes ALL SEVEN curated tables, not two.
# Deploy the GEIB dashboard after it completes — they ship as a pair.
python scripts\export_kalman_analytics.py --dry-run
python scripts\export_kalman_analytics.py

# One-off migration of a pre-0.9.9.13 flat results directory into the section tree.
python pymc_kalman_filter_pt.py --migrate-layout
python pymc_kalman_filter_pt.py --migrate-layout --apply
```

> **Set `PYTHONIOENCODING=utf-8` before redirecting output to a file on Windows.** Console
> stdout falls back to cp1252, and `run_eda` prints `Spearman ρ` (U+03C1), which raises
> `UnicodeEncodeError` mid-run — after minutes of figure rendering. cp1252 *can* encode the
> em-dashes and arrows these scripts print constantly, so most output survives and only the
> Greek glyphs actually fail. `validate_kalman_state.py` prints none, so it redirects
> cleanly and gives false reassurance.

Check the schema vintage at any time — from the notebook or a bare interpreter:

```python
kf.check_export_vintage()   # {table: run_id}, warns when they differ
```

```sql
-- Shipped run 8538a5d6ee25: n_oos = 4, n_blowup = 0, n_vintages = 1
SELECT count(*)                                                 AS n_rows,      -- 6487
       count(*) FILTER (WHERE out_of_support)                   AS n_oos,       -- 4
       count(*) FILTER (WHERE abs(expected_sharpe_ratio) > 1e6) AS n_blowup,    -- 0
       count(DISTINCT run_id)                                   AS n_vintages   -- 1
FROM analytics.kalman_filtered_price_targets;

-- Cross-table vintage check. Was 6,425 of 6,427 before the fix; now 0 of 6,487.
SELECT count(*) FROM analytics."10_screen_results" s
JOIN analytics.kalman_filtered_price_targets k USING (isin)
WHERE abs(s.er_mean - k.er_mean) > 1e-9;
```

Artifacts (PNG / CSV / SQL / JSON / NetCDF) land under `KALMAN_PT_RESULTS_DIR` in the
per-section subdirectories resolved by `_export_dir_for` against the `_EXPORT_SECTION_DIRS`
SSOT — never build a result path by hand. The curated bulk frames in `_SQL_EXPORT_ARTIFACTS`
additionally become `analytics."<stem>"` tables plus a generated `<stem>.sql` DDL file;
`KALMAN_PT_SQL_EXPORT=0` (or an unreachable database) falls back to CSV while still emitting
the DDL. `10c_kalman_results` is suppressed whenever the canonical
`kalman_filtered_price_targets` write has already run this session, so its absence from the
schema is expected, not a gap.

> **The bulk-frame CSVs on disk are not refreshed by a successful SQL export** — the sink
> writes the table and the DDL only. Copies under `04_panel/`, `10_screen/`, `10b_risk/`
> and `10c_analytics/` are leftovers from the last run where SQL export was off or failed.
> Read the `analytics` tables, not those files.

### Still open — Phase B / C

| phase | item |
|---|---|
| **B** | Split `UPLIFT_CLIP_*` into a tight `FIT_*` winsorisation and the wide `MC_*` overflow guard, refit, and re-check the §8 `T = std` statistic. Root cause of all three PPC failures; `nu` is pinned at 2.5259. |
| **C** | Break up the 3-feature PT-history concentration (§3/§9 — the three lowest-ESS betas); replace the thin EPS family with `eps_gaap_est_avg_rev_pct_fy1e_*` (85 % coverage vs 47 %); decide on the dropped `industry`/`country`/`exchange`/`unit` hierarchy levels; add `sigma_isin_level` + `sigma_time_free` to the §9 variance partition; refit §10K/§12 in normalised space. |